# 04 — Quantized continuation setup

Run locally before any 04--06 GPU notebook. Select `streamlined_experiment/.venv-local/Scripts/python.exe` as the kernel. The receiver extends the original service, retains its token and port, and adds atomic row checkpoints plus 25-row CSV snapshots.

## 1. Locate and verify the frozen experiment

Read the completed 00--03 manifest and confirm that the new source code can use it.

**Recovery / safe rerun:** After a local kernel restart, rerun 1. Source and old results are read only.

**Cost:** No paid API requests.



In [ ]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'streamlined/common.py').is_file()), None)
if ROOT is None:
    ROOT = next(p/'streamlined_experiment' for p in candidates if (p/'streamlined_experiment/streamlined/common.py').is_file())
sys.path.insert(0, str(ROOT))
from streamlined.common import *
print('Experiment:', ROOT)
print('Interpreter:', sys.executable)

from streamlined.common import verify_experiment
from continuation.protocol import calibration_rows
print('Experiment:',verify_experiment(ROOT)['config_hash'])
print('Frozen calibration rows:',len(calibration_rows(ROOT)))

## 2. Receiver, HTTPS tunnel and audible watchdog

Run the commands below in external laptop terminals. The original receiver must be stopped before launching the extended receiver on the same port. **Existing active workers should finish before replacing the receiver.**

**Recovery / safe rerun:** If the tunnel or receiver stops, restart the commands and rerun 2. No model rows are repeated.

**Cost:** No paid API requests.

**One-time laptop setup, in separate PowerShell terminals:**

1. Stop the prior `streamlined.receiver` terminal if it is still serving port 8770. Let any active 01--03 worker finish first. Start the extended receiver:

```powershell
Set-Location 'D:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment'
.\.venv-local\Scripts\python.exe -m continuation.receiver --root . --port 8770
```

2. Start the HTTPS tunnel in another terminal and keep it open:

```powershell
cloudflared tunnel --url http://127.0.0.1:8770
```

3. Start the independent audible watchdog in a third terminal:

```powershell
Set-Location 'D:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment'
.\.venv-local\Scripts\python.exe -m streamlined.monitor --root .
```

4. Read the local receiver token and test sound in a fourth terminal:

```powershell
Get-Content .\runtime\receiver_token.txt
.\.venv-local\Scripts\python.exe -m streamlined.monitor --root . --test
```

Use the same URL/token for distinct model notebooks. Never launch the same model and condition twice. Keep Windows awake, audio unmuted, tunnel, receiver and watchdog terminals open. A lost Colab runtime stops its heartbeat; the local watchdog alarms without relying on the notebook kernel.


In [ ]:
from streamlined.transport import Mirror
from streamlined.common import read_json
from pathlib import Path
import urllib.request, json
token = (ROOT/'runtime/receiver_token.txt').read_text().strip()
print('Token file exists:',bool(token))
print('Extended receiver must be running in its own terminal before any Colab bootstrap.')
print('Watchdog state:',read_json(ROOT/'runtime/watchdog_alive.json') if (ROOT/'runtime/watchdog_alive.json').exists() else 'not running')
del token

## 3. Input and expected result counts

Condition A: native BitNet I2_S. Condition B: TQ1_0 for five conventional models. Condition C: optional IQ1_S with only the frozen 604 training prompts as importance-matrix data. Create the exact 604-prompt calibration corpus locally for audit and Colab reuse. Untreated A/B infer 990 rows; every SFT adapter and optional C run infers 140 validation plus 145 test rows.

**Recovery / safe rerun:** Rerun 1 then 3 at any time. No GPU or API work occurs.

**Cost:** No paid API requests.



In [ ]:
from continuation.judge import expected_streams
from continuation.artifacts import calibration_text
corpus,sha = calibration_text(ROOT)
print('Local calibration corpus:',corpus,'SHA256:',sha)
for family in ('required','optional'):
    streams = expected_streams(family)
    print(family, 'candidate streams:',len(streams), 'untreated:',sum(x[2] is None for x in streams),
          'SFT:',sum(x[2] is not None for x in streams))